## Mount Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


## Import Package

In [2]:
!pip install -q sentence-transformers

In [3]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## Load Preprocessed Data

Reads the two saved versions from `01_data_cleaning_and_preprocessing.ipynb`'s Save Preprocess Result section, plus the original complaints file (not saved separately - reloaded and re-cleaned here).

In [4]:
recall_df_option1 = pd.read_csv('data/df_recalls_single_label.csv', low_memory=False)
recall_df_option2 = pd.read_csv('data/df_recalls_keep_multilabel.csv', low_memory=False)
recall_df_option1['RCDATE'] = pd.to_datetime(recall_df_option1['RCDATE'], errors='coerce')
recall_df_option2['RCDATE'] = pd.to_datetime(recall_df_option2['RCDATE'], errors='coerce')

# Already cleaned AND categorized in 01_data_cleaning_and_preprocessing.ipynb - has mk/md/yr, DATEA, and PARENT_GROUP already. No mapping dict needed here anymore.
complaints_df = pd.read_csv('data/df_complaints_categorized.csv', low_memory=False)
complaints_df['DATEA'] = pd.to_datetime(complaints_df['DATEA'], errors='coerce')
complaints_df['PARENT_GROUP'] = complaints_df['PARENT_GROUP'].replace('Unclassified', 'Other')

print("Option 1:", recall_df_option1.shape, " Option 2:", recall_df_option2.shape, " Complaints:", complaints_df.shape)


Option 1: (104966, 34)  Option 2: (163828, 34)  Complaints: (2249816, 56)


In [5]:
recall_df_option1['CAMPNO'].nunique(), recall_df_option2['CAMPNO'].nunique()

(5528, 6180)

## Category List

Both `recall_df_option1`/`recall_df_option2` and `complaints_df` already carry `PARENT_GROUP` from notebook 1 - the category mapping dict itself doesn't need to live here at all anymore. The fixed list of categories is derived directly from the loaded data instead, so this notebook always reflects whatever notebook 1 decided, with nothing duplicated (and nothing that can drift out of sync).

In [6]:
categories = sorted((set(recall_df_option1['PARENT_GROUP']) | set(recall_df_option2['PARENT_GROUP']) | set(complaints_df['PARENT_GROUP'])))
print("Categories:", categories)


Categories: ['ADAS', 'Brakes', 'Chassis', 'Communication', 'Electrical', 'Engine', 'Equipment', 'Fuel', 'Hardware', 'Other', 'Restraints', 'Seating', 'Structure', 'Tires', 'Transmission', 'Vision']


## Build Features 1-8

Shared function for both Option 1 and Option 2 - a campaign's pooled complaint history doesn't depend on how many labels it ends up with, only on its own make/model/year list and recall date.

**Tunable:** `lookback_days` - `None` (default) uses *all* prior complaints, same as before; set it to an integer to restrict to only the past N days before `RCDATE` instead of the full history.

**Output columns:**
- `RCDATE`: the campaign's recall date - the cutoff; every complaint counted below happened strictly before this date.
- `total_prior_complaints`: total number of pooled prior complaints used to build this row.
- `count_<category>` (16 columns, one per category, e.g. `count_Brakes`, `count_Engine`): how many of the prior complaints fall into each category.
- `share_<category>` (16 columns, one per category): the same counts, each divided by `total_prior_complaints`.
- `pct_crash`: share of prior complaints flagging a crash.
- `pct_fire`: share of prior complaints flagging a fire.
- `pct_injured`: share of prior complaints reporting at least one injury.
- `pct_deaths`: share of prior complaints reporting at least one death.
- `trend_recent_minus_prior`: complaints in the last `trend_window_days` days before `RCDATE`, minus complaints in the `trend_window_days` days before that - positive means accelerating, negative means tapering off.
- `glued_text` *(transient only - consumed by Feature 9b below and dropped before saving, never present in the final output)*: the pooled complaint narratives, glued into one string.


In [7]:
def build_campaign_features(recall_table, complaints_df, categories, lookback_days=None, trend_window_days=90):
    campaign_info = recall_table.groupby('CAMPNO').agg(RCDATE=('RCDATE', 'first')).reset_index()
    mmy_lookup = (
        recall_table.groupby('CAMPNO')[['mk', 'md', 'yr']]
        .apply(lambda g: list(set(zip(g['mk'], g['md'], g['yr']))))
        .rename('mmy_list')
    )
    campaign_info = campaign_info.merge(mmy_lookup, on='CAMPNO')

    complaints_sorted = complaints_df.dropna(subset=['DATEA']).sort_values('DATEA')
    grouped_complaints = {key: g for key, g in complaints_sorted.groupby(['mk', 'md', 'yr'])}

    rows = []
    for _, camp in campaign_info.iterrows():
        rcdate = camp['RCDATE']
        pieces = [grouped_complaints[key] for key in camp['mmy_list'] if key in grouped_complaints]
        pool = pd.concat(pieces) if pieces else complaints_sorted.iloc[0:0]

        if pd.notna(rcdate):
            mask = pool['DATEA'] < rcdate
            if lookback_days:
                # Tunable: restrict to the past `lookback_days` days before RCDATE instead
                # of all history. None (default) = no lower bound, same as before.
                mask &= pool['DATEA'] >= (rcdate - pd.Timedelta(days=lookback_days))
            pool = pool[mask]
        else:
            pool = pool.iloc[0:0]

        total = len(pool)
        cat_counts = pool['PARENT_GROUP'].value_counts().reindex(categories, fill_value=0)
        cat_shares = (cat_counts / total) if total > 0 else cat_counts.astype(float) * 0.0

        pct_crash = (pool['CRASH'] == 'Y').mean() if total > 0 else 0.0
        pct_fire = (pool['FIRE'] == 'Y').mean() if total > 0 else 0.0
        pct_injured = (pool['INJURED'] > 0).mean() if total > 0 else 0.0
        pct_deaths = (pool['DEATHS'] > 0).mean() if total > 0 else 0.0

        recent_cut = rcdate - pd.Timedelta(days=trend_window_days)
        prior_cut = rcdate - pd.Timedelta(days=2 * trend_window_days)
        n_recent = (pool['DATEA'] >= recent_cut).sum()
        n_prior = ((pool['DATEA'] >= prior_cut) & (pool['DATEA'] < recent_cut)).sum()

        row = {
            'CAMPNO': camp['CAMPNO'], 'RCDATE': rcdate, 'total_prior_complaints': total,
            'pct_crash': pct_crash, 'pct_fire': pct_fire, 'pct_injured': pct_injured,
            'pct_deaths': pct_deaths, 'trend_recent_minus_prior': n_recent - n_prior,
            'glued_text': " ".join(pool['CDESCR'].dropna().astype(str).tolist()),
        }
        for cat in categories:
            row[f'count_{cat}'] = cat_counts[cat]
            row[f'share_{cat}'] = cat_shares[cat]
        rows.append(row)

    return pd.DataFrame(rows).set_index('CAMPNO')


## Build Feature 9 (Sentence Embeddings)

In [8]:
def add_embedding_features(df, model, text_col='glued_text', prefix='emb_'):
  texts = df[text_col].fillna('').replace('', 'no complaints').tolist()
  embeddings = model.encode(texts, show_progress_bar=False)
  emb_df = pd.DataFrame(embeddings, columns=[f'{prefix}{i}' for i in range(embeddings.shape[1])], index=df.index)
  return pd.concat([df.drop(columns=[text_col]), emb_df], axis=1)

### Option 1 - single-label (one row per campaign)

Every campaign here already has exactly one `PARENT_GROUP`, so attaching `y` is 1:1 - no explode needed.

In [25]:
y_option1 = recall_df_option1.groupby('CAMPNO')['PARENT_GROUP'].first()

features_option1 = build_campaign_features(recall_df_option1, complaints_df, categories)
features_option1['y'] = y_option1

features_option1 = add_embedding_features(features_option1, embedding_model)
print("Option 1 feature table shape:", features_option1.shape)
features_option1.head()


Option 1 feature table shape: (5528, 424)


,RCDATE,total_prior_complaints,pct_crash,pct_fire,pct_injured,pct_deaths,trend_recent_minus_prior,count_ADAS,share_ADAS,count_Brakes,...,emb_374,emb_375,emb_376,emb_377,emb_378,emb_379,emb_380,emb_381,emb_382,emb_383
CAMPNO,,,,,,,,,,,,,,,,,,,,,
10V001000,2010-01-04,94,0.117021,0.106383,0.095745,0.010638,-5,0,0.000000,2,...,0.014331,-0.066681,-0.039147,-0.005420,-0.015744,0.058978,-0.083163,0.004119,0.051914,-0.013554
10V007000,2010-01-08,84,0.071429,0.000000,0.035714,0.000000,6,0,0.000000,4,...,-0.030420,-0.057469,0.061091,-0.006281,0.036242,0.027880,-0.012819,-0.001825,-0.032380,0.044704
10V008000,2010-01-11,1607,0.060361,0.010579,0.050404,0.000622,30,2,0.001245,127,...,-0.025940,0.005885,-0.074856,-0.002802,-0.003537,0.018236,-0.019452,0.084361,-0.035766,0.071527
10V009000,2010-01-11,2,0.000000,0.000000,0.500000,0.000000,1,0,0.000000,0,...,-0.033973,-0.036566,-0.081016,0.036477,0.021791,-0.014724,-0.078592,0.000938,-0.029184,0.012344
10V015000,2010-01-21,3,0.000000,0.000000,0.000000,0.000000,0,0,0.000000,1,...,-0.051348,0.008496,-0.043533,-0.125482,-0.074855,0.043239,-0.003244,0.030854,-0.037767,-0.030633


### Option 2 - multi-label (one campaign becomes multiple rows)

`X` is built once per campaign, exactly like Option 1 - then attached to *every* distinct label that campaign has, via `.explode()`. A 2-label campaign ends up as 2 final rows with identical `X` and different `y` - keep that in mind when training on this version.

In [9]:
labels_option2 = recall_df_option2.groupby('CAMPNO')['PARENT_GROUP'].unique().rename('labels')

features_option2_base = build_campaign_features(recall_df_option2, complaints_df, categories)
features_option2 = features_option2_base.merge(labels_option2, left_index=True, right_index=True)
features_option2 = features_option2.explode('labels').rename(columns={'labels': 'y'})

features_option2 = add_embedding_features(features_option2, embedding_model)
print("Option 2 campaigns:", features_option2_base.shape[0])
print("Option 2 final training rows (after duplicating multi-label campaigns):", len(features_option2))
features_option2.head()


Option 2 campaigns: 6180
Option 2 final training rows (after duplicating multi-label campaigns): 6862


,RCDATE,total_prior_complaints,pct_crash,pct_fire,pct_injured,pct_deaths,trend_recent_minus_prior,count_ADAS,share_ADAS,count_Brakes,...,emb_374,emb_375,emb_376,emb_377,emb_378,emb_379,emb_380,emb_381,emb_382,emb_383
CAMPNO,,,,,,,,,,,,,,,,,,,,,
10V001000,2010-01-04,94,0.117021,0.106383,0.095745,0.010638,-5,0,0.000000,2,...,0.030714,-0.058973,-0.071062,0.025072,0.056408,0.037543,-0.133887,0.015296,0.026205,0.007895
10V007000,2010-01-08,84,0.071429,0.000000,0.035714,0.000000,6,0,0.000000,4,...,-0.032920,-0.052074,0.046254,0.001519,-0.049614,-0.072145,0.039368,-0.033954,-0.021361,0.088930
10V008000,2010-01-11,1607,0.060361,0.010579,0.050404,0.000622,30,2,0.001245,127,...,0.001489,-0.044641,-0.052585,0.110330,-0.075483,-0.060239,-0.074451,0.010978,-0.000020,-0.007669
10V009000,2010-01-11,2,0.000000,0.000000,0.500000,0.000000,1,0,0.000000,0,...,-0.054466,-0.038105,-0.092507,0.035525,0.022108,-0.049793,-0.078763,-0.004900,-0.041359,0.008213
10V015000,2010-01-21,3,0.000000,0.000000,0.000000,0.000000,0,0,0.000000,1,...,-0.042012,-0.023520,-0.062875,-0.115218,-0.015196,0.032555,0.015716,0.035616,0.017384,-0.062215


## Save Feature Engineering Result

In [26]:
features_option1.to_csv('data/features_single_label.csv')
print("Saved Option 1 features:", features_option1.shape)


Saved Option 1 features: (5528, 424)


In [10]:
features_option2.to_csv('data/features_multilabel.csv')
print("Saved Option 2 features:", features_option2.shape)


Saved Option 2 features: (6862, 424)
